In [1]:
import pandas as pd
import json
import datetime
import statistics as stats
from urlextract import URLExtract
import re
from tqdm import tqdm
from urllib.parse import urlparse

url_extractor = URLExtract()

In [2]:
dic_tw = json.load(open('../Data/tweets.json'))

df_notes = pd.read_csv('../Data/notes.tsv', sep='\t', low_memory=False)

def convert_timestamp_tweet(ts):
    return datetime.datetime.strptime(ts, "%a %b %d %H:%M:%S %z %Y")

def convert_timestamp_note(ts):
    seconds = ts / 1000.0

    return datetime.datetime.fromtimestamp(seconds, datetime.UTC)

#### Overall Size

In [3]:
print("The total number of tweets: ", len(dic_tw))

print("The total number of community notes: ", len(df_notes['noteId'].unique()))

ts_tw = [convert_timestamp_tweet(dic_tw[i]['createdAt']) for i in dic_tw]
ts_notes = [convert_timestamp_note(i) for i in df_notes['createdAtMillis']]

print("The time range of tweets: ", min(ts_tw), " to ", max(ts_tw))
print("The time range of community notes: ", min(ts_notes), " to ", max(ts_notes))

The total number of tweets:  78698
The total number of community notes:  169992


The time range of tweets:  2021-01-05 16:06:45+00:00  to  2025-09-06 14:54:12+00:00
The time range of community notes:  2021-01-28 16:57:13.899000+00:00  to  2025-09-06 17:41:39.303000+00:00


#### Tweet level

In [4]:
def return_author_followers(dic_tw):
    au_fol = {}
    for i in dic_tw:
        author = dic_tw[i]['author']['userName']
        followers = dic_tw[i]['author']['followers']
        # In case of multiple tweets by same author, take the max followers count
        if author not in au_fol:
            au_fol[author] = followers
        else:
            au_fol[author] = max(au_fol[author], followers)
    return au_fol

author_followers = return_author_followers(dic_tw)
print("The total number of unique users (tweet authors): ", len(author_followers))

print("The median number of followers: ", stats.median(list(author_followers.values())))

def return_tweet_engagement(dic_tw):
    tweet_eng = []
    for i in dic_tw:
        likes = dic_tw[i]['likeCount']
        retweets = dic_tw[i]['retweetCount']
        # replies = dic_tw[i]['replyCount']
        quotes = dic_tw[i]['quoteCount']
        tweet_eng.append( likes + retweets + quotes )
    return stats.median(tweet_eng)

print("The median tweet engagement (likes + retweets + quotes): ", return_tweet_engagement(dic_tw))

print("The total % of tweets with multiple CNs: ", 100 * len(df_notes['tweetId'].value_counts()[df_notes['tweetId'].value_counts() > 1]) / len(df_notes['tweetId'].unique()) )

The total number of unique users (tweet authors):  19537
The median number of followers:  22423
The median tweet engagement (likes + retweets + quotes):  6216.0
The total % of tweets with multiple CNs:  51.007649495539916


In [7]:
author_names = [dic_tw[i]['author']['userName'] for i in dic_tw]

pd.Series(author_names).value_counts()

elonmusk           965
KamalaHQ           852
POTUS46Archive     801
JoeBiden           613
LeadingReport      570
                  ... 
RepMoBrooks          1
TRUMP4FEDPRISON      1
Spinachbrah          1
stranahan            1
ZakkenKloot          1
Name: count, Length: 19537, dtype: int64

#### Notes contributors

In [5]:
print("Unique number of note writers: ", len(df_notes.noteAuthorParticipantId.unique()))

print("Median number of notes by writers:\n", df_notes.noteAuthorParticipantId.value_counts().describe())

Unique number of note writers:  55386
Median number of notes by writers:
 count    55386.000000
mean         3.069223
std         11.116419
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max       1355.000000
Name: count, dtype: float64


#### Notes helpfulness

In [6]:
print("Currently rated helpfulness level status distribution (%):")
print(df_notes["helpfulnessStatus"].value_counts(normalize=True) * 100, end="\n\n")

print("Total number of raters stats:")
print(df_notes["totalRatings"].describe())

Currently rated helpfulness level status distribution (%):
helpfulnessStatus
NEEDS_MORE_RATINGS             92.246273
CURRENTLY_RATED_NOT_HELPFUL     4.187073
CURRENTLY_RATED_HELPFUL         3.566654
Name: proportion, dtype: float64

Total number of raters stats:
count    169992.000000
mean        149.954492
std         292.262329
min           0.000000
25%          15.000000
50%          55.000000
75%         162.000000
max       12459.000000
Name: totalRatings, dtype: float64


#### Notes level

In [7]:
def notes_text_map(df_notes):
    dic_n = {}
    for id, summ in tqdm(zip(df_notes.noteId, df_notes.summary), total=len(df_notes)):
        urls = url_extractor.find_urls(summ)
        domains = [urlparse(u).netloc for u in urls]
        summ_cleaned = summ
        for u in urls:
            summ_cleaned = summ_cleaned.replace(u, "")

        summ_cleaned = summ_cleaned.replace("\n", " ")
        summ_cleaned = re.sub(r"\s+", " ", summ_cleaned)
        summ_cleaned = summ_cleaned.strip()

        dic_n[id] = {"note": summ, "urls": urls, "domains": domains, "notes_clean": summ_cleaned}
    return dic_n

dic_notes = notes_text_map(df_notes)

100%|██████████| 169992/169992 [05:38<00:00, 502.12it/s]


In [8]:
notes_wrd_cnt = [ len(dic_notes[i]['notes_clean'].split(" ")) for i in dic_notes]

print("Avr (SD) of words in CNs: ", stats.mean(notes_wrd_cnt), ";", stats.stdev(notes_wrd_cnt))

print("\nThe % of number of URLs in CNs:\n", pd.Series([ len(dic_notes[i]['urls']) for i in dic_notes]).value_counts(normalize=True) * 100 , end="\n\n")

domains_notes = [j for i in dic_notes for j in dic_notes[i]['domains']] 

print("The total number of unique domain refered: ", len(set(domains_notes)))

print("\nValue count of domains mentioned in CNs:\n", pd.Series(domains_notes).value_counts(normalize=True) * 100)

Avr (SD) of words in CNs:  28.150871805732034 ; 13.140041081705645

The % of number of URLs in CNs:
 1     54.689044
0     22.934020
2     13.437103
3      5.323192
4      1.945386
5      0.821803
6      0.376488
7      0.188832
8      0.110005
9      0.060591
10     0.034119
11     0.025884
12     0.018824
13     0.005883
17     0.005294
15     0.004706
14     0.003530
16     0.003530
18     0.002941
19     0.002353
23     0.001765
20     0.001177
21     0.001177
25     0.001177
27     0.000588
28     0.000588
Name: proportion, dtype: float64

The total number of unique domain refered:  13238

Value count of domains mentioned in CNs:
 x.com                                7.867028
en.wikipedia.org                     2.871953
apnews.com                           2.696295
en.m.wikipedia.org                   2.612528
www.reuters.com                      2.046463
                                       ...   
kaironews.com                        0.000508
polidiotic.com                    

#### Missingness

In [9]:
def notes_without_urls(dic_notes):
    cnt = 0
    for i in dic_notes:
        if len(dic_notes[i]['urls'])==0:
            cnt += 1
    return cnt

print("Total % of notes without URLs: ", 100*notes_without_urls(dic_notes)/len(dic_notes))

Total % of notes without URLs:  22.934020424490566


#### Temporal analysis

In [10]:
tw_note_time = {}
for tid in tqdm(dic_tw):
    tt = convert_timestamp_tweet( dic_tw[tid]["createdAt"] )
    nt = convert_timestamp_note( df_notes[df_notes.tweetId == int(tid)].createdAtMillis.min() )
    tw_note_time[tid] = nt - tt

print("Descriptive statistics of time lag b/w tweet and first note:\n", pd.Series(tw_note_time.values()).describe())

notes_month_wise = [f"{i.year}-{i.month}"for i in df_notes['createdAtMillis'].map(convert_timestamp_note)]

print("\nMonth wise num of notes stats:\n", pd.Series(notes_month_wise).value_counts().describe())

100%|██████████| 78698/78698 [00:27<00:00, 2911.76it/s]


Descriptive statistics of time lag b/w tweet and first note:
 count                         78698
mean      2 days 17:38:49.309068718
std      33 days 13:29:00.852172821
min          0 days 00:00:22.923000
25%          0 days 01:43:53.163250
50%          0 days 05:38:57.487500
75%          0 days 15:58:26.896000
max       1460 days 23:11:00.386000
dtype: object

Month wise num of notes stats:
 count       57.000000
mean      2982.315789
std       3376.860499
min        158.000000
25%        303.000000
50%       2085.000000
75%       4317.000000
max      15071.000000
Name: count, dtype: float64


In [11]:
df_note_status = pd.read_csv("../Data/Original CN/StatusHistory/noteStatusHistory-00000.tsv", sep="\t", low_memory=False)

df_note_status = df_note_status[df_note_status.noteId.isin(df_notes.noteId)]

df_note_status.index = df_note_status.noteId

note_non_nmr_time = {}
for i in tqdm(range(len(df_notes))):
    tmp = df_notes.iloc[i]
    nt = convert_timestamp_note( tmp.createdAtMillis )
    try:
        st = convert_timestamp_note( df_note_status.loc[tmp.noteId].timestampMillisOfFirstNonNMRStatus )
        note_non_nmr_time[tmp.noteId] = st-nt
    except:
        continue

print("Descriptive statistics of time lag b/w note and first non-NMR status:\n", pd.Series(note_non_nmr_time.values()).describe())

100%|██████████| 169992/169992 [00:16<00:00, 10596.49it/s]

Descriptive statistics of time lag b/w note and first non-NMR status:
 count                        20031
mean     1 days 14:39:45.858197943
std      8 days 11:07:43.165657670
min         0 days 00:10:17.803000
25%         0 days 02:26:32.728000
50%         0 days 05:31:57.706000
75%         0 days 13:09:14.716000
max       194 days 13:58:36.621000
dtype: object
